# A system with no data science in it

If this model only fits problems that look like pipelines over data, it is a
data tool with ambitions. So here is a shipping-notification service: an event
arrives, it gets authenticated, deduplicated, enriched, checked against policy,
rendered and delivered, and the delivery is recorded.

There is no dataset, no model and no score. What there *is*: exactly one step
with an outward effect, and a permission surface that has to be visible before
anything runs.

In [1]:
# Installed from the repository, not from PyPI: this notebook uses `templates`
# and `viz`, which no published release contains yet. A notebook that installs
# something older than the API it calls fails at cell one, which is a confusing
# way to introduce a library about checking things before they run.
try:
    import browsergraph  # noqa: F401
except ImportError:  # pragma: no cover
    %pip install -q "browsergraph @ git+https://github.com/aidonerightcorp/browsergraph.git"

import browsergraph as bg
from browsergraph import templates as T, viz
from browsergraph.compile import CompileError, compile_route
from browsergraph.manifest import NodeManifest, ParameterSpec, PortSpec
from browsergraph.workbench import NodeCandidate

print("browsergraph", bg.__version__)

browsergraph 0.3.0


In [2]:
def node(node_id, capability, ins, outs, *, effects=(), permissions=(),
         params=None, facets=None, deterministic=True, kind="function"):
    """A node manifest in one line, because the notebook is about graphs.

    Real packs write these as JSON in a registry; the shape is the same.
    """
    return NodeManifest(
        id=node_id, kind=kind, description=f"{capability} via {node_id}",
        capabilities=(capability,),
        inputs=tuple(PortSpec(n, t) for n, t in ins),
        outputs=tuple(PortSpec(n, t) for n, t in outs),
        parameters=tuple(params or ()),
        effects=tuple(effects), permissions=tuple(permissions),
        runtime={"deterministic": deterministic},
        facets=dict(facets or {}),
    )

In [3]:
template = T.get("service.notification")
for warning in template.anti_patterns:
    print("•", warning, "\n")

• Deduplicating after enrichment. The expensive lookup runs on every retry, and retries are the normal case, not the exception. 

• Deciding policy inside the renderer. 'Do not send' then depends on a template, and nobody can answer why a message went out. 

• Retrying delivery without an idempotency key. At-least-once plus retry is how one event becomes eleven text messages. 

• Treating the audit record as optional because it has no user-visible effect. It is the only evidence the policy was applied at all. 



In [4]:
nodes = [
    node("svc.receive.webhook", "net.receive", [], [("out", "Event")],
         permissions=("net.listen",)),
    node("svc.receive.queue",   "net.receive", [], [("out", "Event")],
         permissions=("queue.consume",)),

    node("svc.auth.hmac",   "auth.verify",   [("in", "Event")], [("out", "Event")],
         permissions=("secret.read",)),
    node("svc.auth.jwt",    "auth.verify",   [("in", "Event")], [("out", "Event")],
         permissions=("secret.read",)),

    node("svc.dedupe.redis","state.dedupe",  [("in", "Event")], [("out", "Event")],
         permissions=("state.read", "state.write")),

    node("svc.enrich.db",   "data.lookup",   [("in", "Event")], [("out", "Context")],
         permissions=("db.read",)),

    node("svc.policy.rules","policy.decide", [("in", "Context")], [("out", "Decision")]),
    node("svc.policy.quiet","policy.decide", [("in", "Context")], [("out", "Decision")],
         facets={"purpose.statement": "suppress deliveries outside waking hours"}),

    node("svc.render.mjml", "render.template", [("in", "Decision")], [("out", "Message")]),

    # The only nodes that reach outside. Everything above is reversible.
    node("svc.deliver.sms",   "net.send", [("in", "Message")], [("out", "Receipt")],
         effects=("network.write", "user.notified"),
         permissions=("net.send", "pii.read"), deterministic=False),
    node("svc.deliver.email", "net.send", [("in", "Message")], [("out", "Receipt")],
         effects=("network.write", "user.notified"),
         permissions=("net.send", "pii.read"), deterministic=False),

    node("svc.record.append", "state.write", [("in", "Receipt")], [("out", "Audit")],
         effects=("state.write",), permissions=("state.write",)),
]

filling = {
    "receive":      ["svc.receive.webhook", "svc.receive.queue"],
    "authenticate": ["svc.auth.hmac", "svc.auth.jwt"],
    "deduplicate":  ["svc.dedupe.redis"],
    "enrich":       ["svc.enrich.db"],
    "policy":       ["svc.policy.rules", "svc.policy.quiet"],
    "render":       ["svc.render.mjml"],
    "deliver":      ["svc.deliver.sms", "svc.deliver.email"],
    "record":       ["svc.record.append"],
}

bench = template.instantiate(filling)
bench = bench.__class__(**{**bench.__dict__, "nodes": tuple(nodes)})
print("routes:", bench.route_count())

routes: 16


In [5]:
viz.dag(bench)

Figure(svg='<svg viewBox="0 0 1590 226" width="1590" height="226" style="max-width:none" role="img"><defs><marker id="bg57915242-arrow" viewBox="0 0 10 10" refX="9" refY="5" markerWidth="7" markerHeight="7" orient="auto-start-end"><path d="M0,0 L10,5 L0,10 z" fill="#8a93a0"/></marker></defs><text x="153.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 0</text><g><rect x="60" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="69" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Receive event</text><text x="69" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="363.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 1</text><g><rect x="270" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="279" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Authenticate</text><text x="279" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="573.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 2</text><g><rect x="480" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="489" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Deduplicate</text><text x="489" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="783.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 3</text><g><rect x="690" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="699" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Enrich</text><text x="699" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="993.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 4</text><g><rect x="900" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="909" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Apply policy</text><text x="909" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="1203.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 5</text><g><rect x="1110" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1119" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Render message</text><text x="1119" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><text x="1413.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 6</text><g><rect x="1320" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1329" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Deliver</text><text x="1329" y="108.0" font-size="9.5" fill="#68737f">2 candidates</text></g><text x="1623.0" y="44" text-anchor="middle" font-size="10" fill="#68737f">layer 7</text><g><rect x="1530" y="74.0" width="186" height="52" rx="7" fill="#eef1f5" stroke="#8a93a0" stroke-width="1"/><text x="1539" y="93.0" font-size="11.5" font-weight="700" fill="#22303f">Record</text><text x="1539" y="108.0" font-size="9.5" fill="#68737f">1 candidate</text></g><path d="M246,100.0 C258.0,100.0 258.0,100.0 270,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg57915242-arrow)"/><path d="M456,100.0 C468.0,100.0 468.0,100.0 480,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg57915242-arrow)"/><path d="M666,100.0 C678.0,100.0 678.0,100.0 690,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg57915242-arrow)"/><path d="M876,100.0 C888.0,100.0 888.0,100.0 900,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg57915242-arrow)"/><path d="M1086,100.0 C1098.0,100.0 1098.0,100.0 1110,100.0" fill="none" stroke="#8a93a0" stroke-width="1.4" opacity=".75" marker-end="url(#bg57915242-arrow)"/><path d="M1296,100.

## What does this plan actually do to the world?

The question you want answered *before* running anything, and the one a
hand-written script cannot answer at all.

In [6]:
route = {"receive": "svc.receive.webhook", "authenticate": "svc.auth.hmac",
         "deduplicate": "svc.dedupe.redis", "enrich": "svc.enrich.db",
         "policy": "svc.policy.quiet", "render": "svc.render.mjml",
         "deliver": "svc.deliver.sms", "record": "svc.record.append"}

plan = compile_route(bench, route)
print("permissions required:", plan.permissions)
print("effects:             ", plan.effects)
print()
for step in plan.steps:
    mark = "  <-- reaches outside" if step.effects else ""
    print(f"  {step.stage:<13} {', '.join(step.effects) or 'pure':<28}{mark}")

permissions required: ('db.read', 'net.listen', 'net.send', 'pii.read', 'secret.read', 'state.read', 'state.write')
effects:              ('network.write', 'state.write', 'user.notified')

  receive       pure                        
  authenticate  pure                        
  deduplicate   pure                        
  enrich        pure                        
  policy        pure                        
  render        pure                        
  deliver       network.write, user.notified  <-- reaches outside
  record        state.write                   <-- reaches outside


One step out of eight touches the world. Everything before it is replayable, and
that is a fact about the *graph*, available without reading a line of the
implementation.

## Dry-running is a property of the plan

A plan whose only effectful step is the delivery can be executed up to that step
safely. That is not a convention someone has to honour — it is readable from the
effects, so a harness can enforce it.

In [7]:
last_pure = [s.stage for s in plan.steps if not s.effects]
print("safe to run without touching the world:", last_pure)
print("requires authorisation:",
      [s.stage for s in plan.steps if s.effects])

safe to run without touching the world: ['receive', 'authenticate', 'deduplicate', 'enrich', 'policy', 'render']
requires authorisation: ['deliver', 'record']


## Policy refuses before scoring gets a say

`pii.read` is a permission, not a preference. A harness that filters on
permissions removes candidates from the space entirely — they never reach the
part of the system that ranks things, so no amount of a good prior or a
flattering embedding can reintroduce them.

In [8]:
allowed = {"net.listen", "queue.consume", "secret.read", "state.read",
           "state.write", "db.read", "net.send"}          # no pii.read

for candidate in bench.candidates:
    manifest = bench.nodes_by_id[candidate.node_id]
    missing = set(manifest.permissions) - allowed
    if missing:
        print(f"  refused {candidate.id:<22} needs {sorted(missing)}")

  refused svc.deliver.sms        needs ['pii.read']
  refused svc.deliver.email      needs ['pii.read']


Both delivery nodes are refused, so *every* route is refused — which is the
correct and useful answer. A system that quietly picked a lower-scoring legal
route here would be hiding that the task cannot be done under this policy.

## The same four pictures, a completely different problem

In [9]:
viz.funnel([
    ("all routes",       bench.route_count()),
    ("type-legal",       bench.route_count()),
    ("policy-eligible",  0),
], title="notification service — under a policy with no pii.read")

Figure(svg='<svg viewBox="0 0 1000 250" width="1000" height="250" style="max-width:none" role="img"><text x="176" y="83" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">all routes</text><rect x="190" y="66" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="83" font-size="11" fill="#22303f">16</text><text x="176" y="129" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">type-legal</text><rect x="190" y="112" width="670.0" height="26" rx="4" fill="#2d6cb5" opacity="0.72" stroke="#2d6cb5" stroke-width="1"/><text x="870.0" y="129" font-size="11" fill="#22303f">16</text><text x="934.0" y="129" font-size="10" fill="#68737f">÷1</text><text x="176" y="175" text-anchor="end" font-size="11.5" font-weight="700" fill="#22303f">policy-eligible</text><rect x="190" y="158" width="2.0" height="26" rx="4" fill="#1f8a4c" opacity="0.22" stroke="#1f8a4c" stroke-width="1"/><text x="202.0" y="175" font-size="11" fill="#22303f">0</text><text x="266.0" y="175" font-size="10" fill="#68737f">→ none</text><text x="190" y="232" font-size="9.5" fill="#68737f">bar length is log-scaled; labels are exact counts</text></svg>', title='notification service — under a policy with no pii.read', note='Every row is a real filter, in order.', width=1000, height=250)

## The point

Three notebooks, three domains that share no vocabulary: features and folds,
text and layout, events and consent. One model, one compiler, one visualiser,
and not a line of domain-specific drawing code.

What differed between them was the *template* and the *nodes*. What stayed the
same was everything that makes the result checkable.